# 파일 검색 및 검색 (File Search and Retrieval)

- **OpenAI 방식의 RAG 구현**

| 구분           | File Search                                              | Retrieval                                            |
| ------------ | -------------------------------------------------------- | ---------------------------------------------------- |
| **핵심 목적**    | 업로드된 파일을 빠르게 찾아서 모델이 참고하도록 돕는 도구                       | 임베딩 기반으로 문서 의미를 이해하고, 가장 관련된 내용을 뽑아주는 시스템          |
| **디자인 철학**   | 도구(Tool) — Responses API 내부에서 모델이 즉시 쓸 수 있도록 설계됨    | 서비스 레벨 API — 임베딩 + 검색을 조합해서 사용자가 직접 구축하는 검색 시스템 |
| **사용 대상**    | Responses API에서 `tools=[{"type": "file_search"}]` 형태로 지정 | 독립적인 `client.vector_stores.search()` 호출로 개발자가 직접 사용  |
| **내부 구조**    | 모델이 도구로 호출 → 백엔드에서 Vector Store 검색 수행                    | 사용자가 직접 Vector Store 생성, 파일 업로드, 검색, 필터링 수행          |
| **통합성**      | 모델 대화형 통합에 최적화 (자동 grounding, citations 등)               | 검색 로직을 직접 제어 가능 (chunk 크기, score threshold, 필터 등)    |
| **유스케이스 예시** | 이 PDF에서 환불 정책 찾아줘 → File Search로 모델이 자동 호출             | 내 FAQ 데이터에서 ‘환불’ 관련 문장 의미 검색 → Retrieval로 직접 쿼리    |

### 용도
단순 모델 대화에서 문서 참조만 원하면 `file_search`, 검색 시스템 구축이면 `retrieval` 사용

### 벡터 스토어 (Vector stores)

Vector store(벡터 스토어) 는 Retrieval API 와 File Search 도구에서 의미 기반 검색(Semantic Search) 을 가능하게 하는 핵심 컨테이너입니다. 
파일을 벡터 스토어에 추가하면, 해당 파일은 자동으로 청크 단위로 분할(chunked) 되고, 임베딩 (embedding) 이 생성되며, 인덱싱(indexing) 되어 검색 가능한 형태로 저장됩니다.  

- 구조 및 구성 요소
    - 벡터 스토어는 내부적으로 vector_store_file 객체들을 포함합니다.
    - 이 객체들은 실제로는 file 객체를 기반으로 만들어진 래퍼(wrapper) 타입입니다.
    - 즉, file → 실제 업로드된 원본 파일, vector_store.file → 해당 파일을 임베딩/인덱싱하여 벡터 스토어에 등록한 형태

| 객체 유형 (Object type)   | 설명 (Description)                                                                                                                                                  |
| --------------------- | ----------------------------------------------------------------------------------------------------------------------------------------------------------------- |
| **file**              | Files API를 통해 업로드된 콘텐츠를 나타냅니다.                                                              |
| **vector_store**      | 검색 가능한 파일들을 담는 **컨테이너(Container)** 입니다.                                                                                                                           |
| **vector_store.file** | 벡터 스토어에 연결된 개별 파일을 나타내는 **래퍼(wrapper) 타입**입니다.<br>해당 파일은 이미 **청크 분할**, **임베딩**, **인덱싱** 처리가 완료된 상태입니다.<br>또한, **속성(attribute) 기반 필터링**을 위한 `attributes` 맵을 포함합니다. |

- 가격
| **저장 용량 (Storage)**      | **요금 (Cost)**      |
| ------------------------ | ------------------ |
| **총 1GB 이하 (모든 스토어 합산)** | **무료 (Free)**      |
| **1GB 초과분**              | **하루당 $0.10 / GB** |


## 파일 검색 (File Search)
모델이 응답을 생성하기 전에 관련 정보를 파일에서 검색할 수 있도록 합니다.
파일 검색은 Responses API에서 사용할 수 있는 도구입니다. 모델이 시맨틱 및 키워드 검색을 통해 이전에 업로드된 파일의 지식 기반에서 정보를 검색할 수 있도록 합니다. 벡터 저장소를 생성하고 파일을 업로드하면, 모델이 이러한 지식 기반 또는 벡터 저장소에 접근할 수 있도록 하여 모델의 내재된 지식을 강화할 수 있습니다.  

OpenAI에서 관리하는 호스팅 도구이므로 실행을 처리하기 위해 직접 코드를 구현할 필요가 없습니다. 모델이 이 도구를 사용하기로 결정하면 자동으로 도구를 호출하고 파일에서 정보를 검색하여 출력을 반환합니다.

Responses API로 파일 검색을 사용하려면 먼저 벡터 저장소에 지식 기반을 설정하고 파일을 업로드해야 합니다.  

지식 기반을 설정한 후에는 모델에서 사용할 수 있는 도구 목록에 file_search 도구를 포함할 수 있으며, 검색할 벡터 저장소 목록도 포함할 수 있습니다.

In [1]:
from dotenv import load_dotenv
load_dotenv() 

True

In [2]:
from openai import OpenAI
client = OpenAI()

Model = "gpt-5-mini"

1. 벡터 저장소를 만들고 파일을 업로드합니다.

In [3]:
vector_store = client.vector_stores.create(        # vector store 생성
    name="Support FAQ",
)

client.vector_stores.files.upload_and_poll(        # file 업로드
    vector_store_id=vector_store.id,
    file=open("재무제표/네이버.pdf", "rb")
)

VectorStoreFile(id='file-9yfoJ9AYCdCr8rjQzt3GcU', created_at=1760263736, last_error=None, object='vector_store.file', status='completed', usage_bytes=35278, vector_store_id='vs_68eb7e3607608191a2b70fd9e466f8ce', attributes={}, chunking_strategy=StaticFileChunkingStrategyObject(static=StaticFileChunkingStrategy(chunk_overlap_tokens=400, max_chunk_size_tokens=800), type='static'))

2. 검색어를 보내 관련 결과를 얻습니다.

In [4]:
resp = client.responses.create(
    model=Model,
    instructions="제공된 출처를 바탕으로 질의에 대한 간결한 답변을 작성하세요.",
    input="네이버의 목표주가는 얼마인가요?",
    tools=[{
        "type": "file_search",
        "vector_store_ids": [vector_store.id]
    }]
)

print(resp.output_text)

한화투자증권 리서치(공표일 2024-02-05)는 네이버(NAVER)의 목표주가를 280,000원으로 제시했습니다. 현재(공표일 기준) 주가는 221,500원으로 표기되어 있습니다 .

다른 증권사 목표가나 최신 시세도 확인해 드릴까요?


- 응답에 검색 결과 포함
  
출력 텍스트에서 주석(파일 참조)을 볼 수 있지만, 파일 검색 호출은 기본적으로 검색 결과를 반환하지 않습니다.  
응답에 검색 결과를 포함하려면 응답을 생성할 때 include 매개변수를 사용할 수 있습니다.  

In [5]:
resp = client.responses.create(
    model=Model,
    instructions="제공된 출처를 바탕으로 질의에 대한 간결한 답변을 작성하세요.",
    input="네이버의 목표주가는 얼마인가요?",
    tools=[{
        "type": "file_search",
        "vector_store_ids": [vector_store.id]
    }],
    include=["file_search_call.results"]
)

print(resp.output_text)
print()
# print(resp.output)

한화투자증권 리서치는 2024년 2월 5일자 리포트에서 네이버의 목표주가를 280,000원으로 제시했습니다. 

다른 기관의 목표주가나 최신 시세도 확인해 드릴까요?



### 다수 파일의 vector store 생성

In [6]:
# "Financial Statements"라는 벡터 스토어 생성
vector_store = client.vector_stores.create(name="Financial Statements")
 
# OpenAI에 업로드할 파일 준비
file_paths = ["재무제표/goog-10k.pdf", "재무제표/brka-10k.pdf"]
file_streams = [open(path, "rb") for path in file_paths]
 
# 업로드 및 폴링 SDK 도우미를 사용하여 파일을 업로드하고 벡터 스토어에 추가,
# 파일 배치의 완료 상태를 폴링합니다.
file_batch = client.vector_stores.file_batches.upload_and_poll(
  vector_store_id=vector_store.id, files=file_streams
)
 
# 이 작업의 결과를 보기 위해 상태 및 파일 개수를 출력할 수 있습니다.
print(file_batch.status)
print(file_batch.file_counts)

completed
FileCounts(cancelled=0, completed=2, failed=0, in_progress=0, total=2)


In [9]:
response = client.responses.create(
    model=Model,
    input=[
        {"role": "user", "content": "2023년 10월 말에 AAPL의 발행 주식 수는 얼마였나요?"},
        {"role": "user", "content": "2023년 Google의 당기 순이익은 얼마였나요? BERKSHIRE HATHAWAY INC. 와 어느 쪽의 당기 순이익이 더 많았나요?"}
    ],
    tools=[{
        "type": "file_search",
        "vector_store_ids": [vector_store.id]
    }]
)

print(response.output_text)

좋습니다 — 확인해보니 현재 업로드된 파일들에서는 버크셔(제출된 10‑K)는 검색되어 2020년 등 과거 연도 수치가 들어있습니다(예: Berkshire의 2020년 순이익 관련 수치) . 그러나 사용자가 요청하신
- AAPL의 “2023년 10월 말 발행 주식 수”,
- 2023년 Google(Alphabet)의 당기순이익,
- 그리고 2023년 기준으로 Berkshire Hathaway와의 당기순이익 비교

에 대한 최신(2023년) 수치는 업로드된 문서들에서 찾을 수 없습니다.

진행 방식 중 어느 쪽으로 할지 알려주세요:
1) 제가 외부(웹)에서 공식 공시(10‑K/10‑Q)와 신뢰 가능한 소스(SEC, 회사보고서 등)를 찾아 2023년 수치들을 조회해 드리기 — (권한 확인 후 바로 검색).  
2) 사용자가 Apple/Alphabet/Berkshire의 관련 2023 공시 파일을 업로드해 주시면, 그 문서들에서 정확한 수치를 찾아서 인용 포함해 답변드리기.

또 한 가지 확인 부탁드립니다:
- “Google”은 Alphabet Inc.을 의미하는 것으로 보이는데 괜찮으신가요?
- 당기순이익은 GAAP 기준(연간 Consolidated net income)으로 드리면 될까요?
- 결과는 달러(USD)로 표기해 드려도 괜찮겠습니까?

원하시는 옵션(1 또는 2)과 위 확인사항을 알려주시면 바로 처리하겠습니다.


In [10]:
# 새 벡터 스토어 생성
vector_store = client.vector_stores.create(name="기업분석보고서")
 
# OpenAI에 업로드할 파일 준비
file_paths = ["재무제표/네이버.pdf", "재무제표/LG엔솔.pdf"]
file_streams = [open(path, "rb") for path in file_paths]
 
# 업로드 및 폴링 SDK 도우미를 사용하여 파일을 업로드하고 벡터 스토어에 추가,
# 파일 배치의 완료 상태를 폴링합니다.
file_batch = client.vector_stores.file_batches.upload_and_poll(
  vector_store_id=vector_store.id, files=file_streams
)
 
# 이 작업의 결과를 보기 위해 상태 및 파일 개수를 출력할 수 있습니다.
print(file_batch.status)
print(file_batch.file_counts)

completed
FileCounts(cancelled=0, completed=2, failed=0, in_progress=0, total=2)


In [11]:
response = client.responses.create(
    model=Model,
    input=[
        {"role": "user", "content": "LG에너지솔루션의 2024년 하반기 전망을 요약해 주세요. 에코프로비엠은 매출액이 줄은 것에 비해 왜 영업이익이 더 크게 줄어들었나요?"},
        {"role": "user", "content": "2023년 4분기의 네이버의 영업이익은 얼마였나요? 한화 투자 증권이 목표가를 상향 조정한 이유는 무엇인가요?"}
    ],
    tools=[{
        "type": "file_search",
        "vector_store_ids": [vector_store.id]
    }]
)

print(response.output_text)

요청하신 항목별로 요약·응답드립니다.

1) LG에너지솔루션 — 2024년 하반기(요약)
- 전반적 전망: 삼성증권 리포트는 2024년 연간으로 매출 28,187십억원, 영업이익 2,000십억원(2023 대비 소폭 하향)을 제시하면서, 2025~2026년에는 이익의 큰 회복을 예상하고 있습니다.   
- 단기(2분기~상반기) 이슈: 2Q24 실적은 컨센서스에 미달할 것으로 전망되며(2Q 영업이익 전망치가 컨센서스보다 낮음), 주요 원인은 전기차 고객향 수요 둔화, 제품 믹스에서 LFP 비중 확대(ASP 하락 압력), 그리고 ESS 부문의 적자 영향 등입니다. 이로 인해 2분기 영업이익률이 일시적으로 낮아질 것으로 리포트는 분석합니다.   
- 중기 포인트: 반등 시점은 자동차 전지 판매 회복 및 원자재/ASP 안정화, 그리고 2025년 이후 수요 회복에 따른 이익률 개선에서 나타날 것으로 보고 있습니다. 

2) 에코프로비엠(에코프로비엠) — 매출 감소 대비 영업이익의 더 큰 감소 원인
- 현재 제공된 업로드 파일에서는 에코프로비엠 관련 리포트/재무자료를 찾지 못했습니다. 업로드 파일이 있다면 해당 파일을 알려주시거나 업로드해 주세요. 없을 경우 공개 자료(뉴스/애널리스트 리포트 등)를 검색해 원인(예: 원가 상승, 고정비 부담 확대, 제품 믹스 악화, 일회성 비용 등)을 근거와 함께 분석해 드리겠습니다.

3) 네이버 — 2023년 4분기 영업이익과 한화투자증권의 목표가 상향 이유
- 2023년 4분기 영업이익: 4,055억 원을 기록했습니다.   
- 한화투자증권이 목표주가(280,000원)로 상향한 주요 이유: 웹툰 지분가치 상향 반영, 광고·커머스 부문의 수익성 개선(비용통제에 따른 이익 개선) 반영, 그리고 이로 인해 2024년 영업이익 전망치를 약 4% 상향 조정한 점 등을 근거로 제시했습니다. 또한 AI 기반 광고/커머스 및 웹툰(엔터) 상장 가능성 등에서 추가적 업사이드가 기대된다는 점을 언급했습니다. 

원하시면:
- 에코프로비엠에 대해(업로드

# 검색 (Retrieval)

- 데이터를 의미적 유사도(semantic similarity) 기반으로 검색하기

### 시맨틱 검색 (Semantic Search)

시맨틱 검색(Semantic Search) 은 벡터 임베딩(Vector Embeddings) 을 활용하여 의미적으로 연관된 결과를 찾아내는 검색 기법입니다.
이 방식의 핵심은, 단순히 키워드가 일치하지 않아도 의미가 유사한 결과를 찾을 수 있다는 것입니다.
즉, 기존의 키워드 기반 검색으로는 놓칠 수 있는 결과를 포착할 수 있습니다.

- 예시
검색 문장: “우리가 달에 간 게 언제였지?”  
가능한 결과 예시는 다음과 같습니다.
| 문장(텍스트)                    | 키워드 유사도 | 의미 유사도 |
| -------------------------- | ------- | ------ |
| 첫 번째 달 착륙은 1969년 7월에 일어났다. | 0%      | 65%    |
| 달에 간 첫 번째 사람은 닐 암스트롱이었다.   | 27%     | 43%    |
| 내가 달 케이크를 먹었을 때, 정말 맛있었다.  | 40%     | 28%    |


Retrieval API는 데이터에 대해 의미 기반 검색(Semantic Search) 을 수행할 수 있게 해줍니다.
이는 단순히 키워드가 일치하는 결과가 아니라, 의미적으로 유사한 정보를 찾아주는 기술입니다.  
예를 들어, “환불 정책이 뭐야?”라고 물으면 “반품 가능 기간은 30일입니다” 같은 문장을 찾아줍니다.  
이 검색 기능은 단독으로도 유용하지만, 모델과 결합하면 매우 강력한 지식 기반 응답 생성 시스템을 만들 수 있습니다.  

Retrieval은 벡터 스토어(Vector Store) 라는 데이터 인덱스를 기반으로 작동합니다.  
문서를 업로드하면 내부적으로 임베딩(embedding) 을 생성하고, 이를 통해 의미적으로 유사한 문서를 빠르게 찾아낼 수 있습니다.

### 벡터 스토어 생성/파일 업로드

In [12]:
vector_store = client.vector_stores.create(        # vector store 생성
    name="Support FAQ",
)

client.vector_stores.files.upload_and_poll(        # file 업로드
    vector_store_id=vector_store.id,
    file=open("data/customer_policy.txt", "rb")
)

VectorStoreFile(id='file-HgoAR83JR2hP7YHn5gDgnF', created_at=1760264217, last_error=None, object='vector_store.file', status='completed', usage_bytes=71147, vector_store_id='vs_68eb8018628c819197059d9b47588863', attributes={}, chunking_strategy=StaticFileChunkingStrategyObject(static=StaticFileChunkingStrategy(chunk_overlap_tokens=400, max_chunk_size_tokens=800), type='static'))

In [13]:
# 검색 쿼리를 전송하여 관련 결과를 가져옵니다.
user_query = "서비스가 중단될 경우 영업상 발생한 손해는 어떻게 처리되나요?"

results = client.vector_stores.search(
    vector_store_id=vector_store.id,
    query=user_query,
)

In [14]:
combined_text = "\n\n---------------\n\n".join(
    c.text for r in results.data for c in r.content if hasattr(c, "text")
)
print(combined_text)

변경. 당사는 수시로 가격을 변경할 수 있습니다. 구독료를 인상할 경우 최소한 30일 전에 사전 통지하며 가격 인상은 귀하의 다음번 갱신부터 효력이 발생되므로 가격인상에 동의하지 않을 경우 구독을 취소할 수 있습니다.

해지 및 중단
해지. 귀하는 언제든지 본 서비스 이용을 중단할 수 있습니다. 당사는 다음과 같이 판단할 경우 귀하의 본 서비스에 대한 이용을 중단하거나 해지 또는 귀하의 계정을 삭제할 권리를 보유합니다:

본 약관 또는 당사의 이용 정책⁠을 위반한 경우.
법률 준수를 위하여 위와 같이 조치하여야 할 경우.
귀하의 본 서비스 이용이 OpenAI, 당사 사용자 또는 타인에게 위험이나 피해를 초래할 수 있을 경우.
귀하의 계정이 1년 이상 휴면 상태이고 유료계정이 없을 경우 귀하의 계정을 해지할 수 있습니다. 위와 같은 경우 사전 통지합니다.

이의 신청. 당사가 귀하의 계정을 실수로 정지하거나 해지하였다고 판단한 경우, 당사 지원팀에 연락⁠(새 창에서 열기)하여 이의를 제기할 수 있습니다.

서비스의 중단
당사는 본 서비스를 중단하기로 결정할 수 있으나, 중단하는 경우, 사전 통지할 것이며 미사용 선불 서비스는 환불됩니다.

보증의 부인
본 서비스는 “있는 그대로” 제공됩니다. 법률상 금지되는 범위를 제외하고, 당사 및 당사의 계열사 및 라이선스 제공자는 본 서비스와 관련하여 어떠한 보장(명시적, 묵시적, 법적 또는 기타)도 하지 않으며, 상업성, 특정 목적 적합성, 만족스러운 품질, 비침해 및 향유에 관한 보장과 거래 과정에서 또는 거래상 이용 과정에서 발생하는 모든 보장을 부인합니다. 당사는 본 서비스가 중단 없이, 정확하고 오류가 없으며, 또는 콘텐츠가 안전하거나 손실 또는 변경되지 않을 것임을 보장하지 않습니다.

---------------

서비스의 중단
당사는 본 서비스를 중단하기로 결정할 수 있으나, 중단하는 경우, 사전 통지할 것이며 미사용 선불 서비스는 환불됩니다.

보증의 부인
본 서비스는 “있는 그대로” 제공됩니다. 법률상 금

### 응답 합성 (Synthesizing responses)
검색 쿼리를 수행한 후에는 그 결과를 바탕으로 응답을 종합(synthesize) 하고 싶을 때가 있습니다.  
이를 위해 OpenAI의 모델을 활용할 수 있으며, 검색 결과(results) 와 원래의 질의(original query) 를 함께 제공하면,  
모델이 해당 자료에 근거한 사실 기반의 응답(grounded response) 을 생성해 줍니다.

In [15]:
response = client.responses.create(
    model=Model,
    instructions="제공된 출처를 바탕으로 질의에 대한 간결한 답변을 작성하세요.",
    input=f"Sources: {combined_text}\n\nQuery: '{user_query}'"
)

print(response.output_text)

간단히 요약하면 다음과 같습니다.

- 서비스 중단 시 절차: 당사는 서비스를 중단하기로 결정하면 사전 통지하며, 미사용 선불요금(서비스 크레딧 등)은 환불합니다.  
- 영업상 손해에 대한 책임: 당사는 이윤·영업권·이용 또는 데이터 손실 등 간접적·결과적·징벌적 손해에 대해 책임을 지지 않습니다.  
- 책임 한도: 당사의 총 책임은 해당 책임이 발생하기 전 12개월 동안 귀하가 해당 서비스에 대해 지급한 금액과 미화 $100 중 더 큰 금액으로 제한됩니다.  
- 예외 사항: 일부 국가·주에서는 이러한 보증 배제 또는 손해배상 제한이 허용되지 않을 수 있으므로, 거주지 법률에 따라 추가적인 권리가 있을 수 있습니다.  
- 조치 권고: 중대한 손해가 발생한 경우 지원팀에 문의하여 이의 제기/해결 절차를 진행하시기 바랍니다(비공식 해결 → 필요 시 중재).

원하시면 귀하의 구체적 상황(유료구독 여부, 선불 크레딧 유무 등)에 따라 예상 환불·대응 방안을 더 구체적으로 안내해 드리겠습니다.
